# 09 | Full-Development Grouped Cross-Validation

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Objective:** Compare the same three classical ML baselines across five subject-held-out folds on all **29 HARTH development participants**. Compare per-participant scores and inspect the paired Random Forest versus Logistic Regression difference.

**Scientific boundary:** This is development-only Cross-Validation for a previously explored model family, not an independent final test, nested CV, prospective validation or a published benchmark. Original reserved Test IDs `S023` and `S032` are never loaded in this notebook.

## 01. Import existing reusable methods

Data acquisition, preprocessing, feature extraction, CV and evaluation remain in documented Python files. Comments identify the purpose of each notebook operation.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Locate the repository and register the shared Python source folders.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root.parent == root:
        raise RuntimeError("Open this notebook inside human-movement-intelligence.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))

# Reuse the validated cohort and CV functions; no custom HMI package needed.
from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, prepare_harth_feature_cohort
from core.data_splitting import audit_group_class_coverage
from core.group_cross_validation import (
    evaluate_group_cross_validation, compare_models_by_subject,
)
print("HARTH full-development CV components: ready")

## 02. Rebuild the 29-person development cohort

The same local source files and time-safe 100-point/50-stride windows are used as Notebook 08. Caches are reused. Codes outside the predefined eight-class task are excluded only after windowing. This step may take several minutes.

In [ ]:
# Fix cohort membership before any model fitting.
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))
assert len(DEVELOPMENT_SUBJECTS) == 29
assert not (set(DEVELOPMENT_SUBJECTS) & RESERVED_TEST)

# Apply the same data preparation policy used in Notebook 08.
features, metadata, overview, excluded = prepare_harth_feature_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES,
    window_size=100, step_size=50, sampling_rate=50,
)
assert len(features) == len(metadata)
print("Development feature matrix:", features.shape)
print("Excluded out-of-task windows:", int(overview["excluded_windows"].sum()))
print("Reserved Test participants loaded: NO")

## 03. Audit every fold before training

GroupKFold must contain no shared subjects between Train and Validation. Check eight-class coverage for every fold before fitting any classifiers.

In [ ]:
# Confirm full coverage and show the group assignments planned by GroupKFold.
coverage, fold_audit = audit_group_class_coverage(
    labels=metadata["label"].to_numpy(),
    group_ids=metadata["subject_id"].to_numpy(),
    classes=TASK_CLASSES, n_splits=5,
)
display(fold_audit[[
    "fold", "train_subjects", "validation_subjects",
    "train_windows", "validation_windows",
    "missing_train_classes", "missing_validation_classes",
]])
assert not fold_audit["missing_train_classes"].map(bool).any()
assert not fold_audit["missing_validation_classes"].map(bool).any()
print("Fold class-coverage audit: PASSED")

## 04. Run the three model families across all five folds

This may take noticeably longer than earlier notebooks. Each fold trains new models using only that fold's Train subjects; Logistic Regression's scaler is refitted only on the current fold's Train observations. No reserved Test data are involved.

In [ ]:
# Train and score the existing baselines in five development folds.
fold_results, subject_results, summary = evaluate_group_cross_validation(
    features=features.to_numpy(),
    labels=metadata["label"].to_numpy(),
    group_ids=metadata["subject_id"].to_numpy(),
    classes=TASK_CLASSES,
    n_splits=5, random_state=42,
)
display(summary.round(4))
display(fold_results[[
    "fold", "model", "validation_subjects",
    "validation_windows", "validation_classes_present",
    "macro_f1", "balanced_accuracy",
]].round(4))

## 05. Visualise variation across the five held-out folds

Fold-level scores can be influenced by unequal sample counts and subject composition. Do not interpret the five folds as independent repetitions of a randomised clinical trial.

In [ ]:
# Compare every model's held-out Macro F1 using the same y-axis.
fig, ax = plt.subplots(figsize=(10, 5))
for model_name, table in fold_results.groupby("model", sort=False):
    table = table.sort_values("fold")
    ax.plot(table["fold"], table["macro_f1"], marker="o",
            label=f"{model_name} (mean={table['macro_f1'].mean():.3f})")
ax.set(title="HARTH: 29-person development grouped CV",
       xlabel="Held-out fold", ylabel="Macro F1",
       ylim=(0, 1), xticks=[1, 2, 3, 4, 5])
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## 06. Compare the two leading models on identical participants

The paired comparison treats each held-out **participant** equally, not thousands of temporally overlapping windows as independent observations. Positive differences favour Random Forest; negative differences favour Logistic Regression.

In [ ]:
# Use fixed eight-class Macro F1 for the same scientific task denominator.
paired, paired_summary = compare_models_by_subject(
    subject_results,
    model_a="Random Forest",
    model_b="Logistic Regression",
    score_column="macro_f1_fixed_task",
)
display(paired.round(4))
display(pd.DataFrame([paired_summary]))

# Show the signed per-subject difference rather than only a global mean.
fig, ax = plt.subplots(figsize=(13, 5))
bar_colours = ["tab:blue" if value >= 0 else "tab:orange"
               for value in paired["difference_a_minus_b"]]
ax.bar(paired["subject_id"], paired["difference_a_minus_b"], color=bar_colours)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Per-subject Macro F1 difference: Random Forest minus Logistic Regression",
       xlabel="Held-out development participant", ylabel="Paired Macro F1 difference")
ax.tick_params(axis="x", rotation=70)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

## 07. Compare class support and participant performance

Individual participants contribute different activity classes. Report support and per-person fixed-task Macro F1 rather than using a single aggregate score to infer generalisation.

In [ ]:
# Summarise participant-level support and model scores without test leakage.
person_support = pd.DataFrame({
    "subject_id": coverage.index,
    "eligible_windows": coverage.sum(axis=1).to_numpy(),
    "observed_classes": (coverage > 0).sum(axis=1).to_numpy(),
})
comparison = subject_results.pivot(
    index="subject_id", columns="model", values="macro_f1_fixed_task"
).reset_index()
participant_table = person_support.merge(comparison, on="subject_id", how="left")
display(participant_table.round(4))
print("Independent development subjects represented:", len(participant_table))

## 08. Research quality gate and limitations

This is a large but still exploratory participant-held-out development experiment. The original reserved Test subjects have not been passed to loading, feature extraction, training or validation in this notebook. Earlier exploratory work did inspect some properties of these Test subjects, so do not claim strict preregistered blindness.

In [ ]:
# Confirm fold-to-subject coverage, label specification and final Test exclusion.
assert len(DEVELOPMENT_SUBJECTS) == len(set(DEVELOPMENT_SUBJECTS)) == 29
assert not (set(DEVELOPMENT_SUBJECTS) & RESERVED_TEST)
assert len(subject_results["subject_id"].unique()) == 29
assert not (set(subject_results["subject_id"]) & RESERVED_TEST)
assert len(fold_results) == 5 * 3
assert len(paired) == 29
assert subject_results.groupby(["model", "subject_id"]).size().eq(1).all()
assert np.isfinite(features.to_numpy()).all()
print("Full-development GroupKFold: PASSED")
print("Original final Test loaded or scored: NO")
print("Final independent benchmark claim: NOT SUPPORTED")